# Speech-to-Text Demo

## 1. Setup

In [ ]:
# Install missing packages
!pip install -q -U transformers accelerate pydub torchaudio gdown jiwer

In [ ]:
import os
import torch
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display
from base64 import b64decode
from pydub import AudioSegment
import io
import json
import wave
import zipfile
import numpy as np
import torchaudio
import jiwer

In [ ]:
_RECORD_JS = """
async function record(ms, prompt) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then <b>${prompt}</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def _to_waveform(segment):
    segment = segment.set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def record_audio(seconds=8, prompt="speak now"):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(prompt)})")
    raw = b64decode(data_url.split(",", 1)[1])
    return _to_waveform(AudioSegment.from_file(io.BytesIO(raw)))

my_recordings = {}
my_transcripts = {}

def save_recording(name, waveform, sr, text=None):
    """Keep a recording (and its known text, if given) in memory for this session - not written to disk or Drive, gone on runtime reset."""
    my_recordings[name] = (waveform, sr)
    if text is not None:
        my_transcripts[name] = text

def clear_recordings():
    """Delete every in-session recording (and its text) from memory - no undo."""
    my_recordings.clear()
    my_transcripts.clear()

def download_recordings():
    """Zip every recording + a transcripts.json (same format as the common bank) and trigger a browser download to your PC."""
    from google.colab import files

    zip_path = "/content/my_recordings.zip"
    with zipfile.ZipFile(zip_path, "w") as zf:
        for name, (waveform, sr) in my_recordings.items():
            pcm16 = np.clip(waveform * 32768, -32768, 32767).astype(np.int16)
            wav_path = f"/content/{name}.wav"
            with wave.open(wav_path, "wb") as wav_file:
                wav_file.setnchannels(1)
                wav_file.setsampwidth(2)
                wav_file.setframerate(sr)
                wav_file.writeframes(pcm16.tobytes())
            zf.write(wav_path, arcname=f"{name}.wav")
        zf.writestr("transcripts.json", json.dumps(my_transcripts, ensure_ascii=False, indent=2))
    files.download(zip_path)

def load_sample(name):
    """Load a clip by name - checks your in-session recordings (Section 3) first, then the common bank (Section 2)."""
    if name in my_recordings:
        return my_recordings[name]
    path = f"{AUDIO_SAMPLES_DIR}/{name}"
    if os.path.exists(path):
        return _to_waveform(AudioSegment.from_file(path))
    raise FileNotFoundError(f"'{name}' not found in your recordings or {AUDIO_SAMPLES_DIR}")

def to_16k(waveform, sr):
    """Whisper expects 16kHz audio."""
    if sr == 16000:
        return waveform
    resampled = torchaudio.functional.resample(
        torch.from_numpy(waveform), orig_freq=sr, new_freq=16000
    )
    return resampled.numpy()

# WER/CER, case- and punctuation-insensitive (raw ASR punctuation/casing
# shouldn't count as an "error" for this demo).
_WER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])
_CER_TRANSFORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfChars(),
])

def word_error_rates(reference, hypothesis):
    """(WER, CER) between a reference transcript and a model hypothesis."""
    wer = jiwer.wer(reference, hypothesis, reference_transform=_WER_TRANSFORM, hypothesis_transform=_WER_TRANSFORM)
    cer = jiwer.cer(reference, hypothesis, reference_transform=_CER_TRANSFORM, hypothesis_transform=_CER_TRANSFORM)
    return wer, cer

## 2. Load audio samples

In [ ]:
# audio samples: a read-only folder of example clips, distributed as a single
# zip (one Drive download instead of one-per-file - much faster)
AUDIO_SAMPLES_DIR = "/content/audio_samples"
AUDIO_SAMPLES_ZIP_ID = "1J89ZSE5tx07ikub6Z03pTITwWcUKg4y2"  # stt_samples.zip

In [ ]:
import shutil

shutil.rmtree(AUDIO_SAMPLES_DIR, ignore_errors=True)
os.makedirs(AUDIO_SAMPLES_DIR, exist_ok=True)

try:
    import gdown
    zip_path = gdown.download(id=AUDIO_SAMPLES_ZIP_ID, output="/content/audio_samples.zip", quiet=False)
    with zipfile.ZipFile(zip_path) as zf:
        # Flatten structure - extract every file to AUDIO_SAMPLES_DIR directly
        for member in zf.namelist():
            if member.endswith("/"):
                continue
            with zf.open(member) as src, open(f"{AUDIO_SAMPLES_DIR}/{os.path.basename(member)}", "wb") as dst:
                dst.write(src.read())
except Exception as e:
    print(f"Couldn't download the common sample bank ({e!r}) — you can still use Section 3 to record your own samples.")

audio_samples = sorted(f for f in os.listdir(AUDIO_SAMPLES_DIR) if f != "transcripts.json")

transcripts_path = f"{AUDIO_SAMPLES_DIR}/transcripts.json"
if os.path.exists(transcripts_path):
    with open(transcripts_path) as f:
        transcripts = json.load(f)
else:
    transcripts = {}

print(f"\n{audio_samples}")
print(f"\n{len(audio_samples)} audio sample(s) loaded.")
print(f"\n{len(transcripts)} reference transcript(s) loaded.")

In [ ]:
# Preview all samples.
for name in audio_samples:
    print(f"\n{name}\n{transcripts[name]}")
    display(Audio(f"{AUDIO_SAMPLES_DIR}/{name}"))

## 3. Create new samples (yours only)

Record as many named clips as you like. Set `sample_name` (a short id you'll use to refer to it later) and `text` (exactly what you're about to read aloud — lets Section 5 print WER/CER automatically), then run the cell: allow microphone access when prompted, click **Start**, and read the prompt out loud. Re-run with different `sample_name`/`text` values to record more — a numbers example, code-switching, whatever you want to test live.

The two cells below default to `english_example`/`slovak_example` — the same lines OmniVoice's TTS demo speaks — so Section 5's example cells work out of the box; record at least those two once, then add whatever else you like.

Everything here is kept in memory for this session only (`my_recordings`/`my_transcripts`, from Section 1) — nothing is written to disk or Drive, and it's gone once the runtime resets.

In [ ]:
sample_name = "english_example"
text = "Hello NLP Summer School of 2026. Welcome in Kinit."

waveform, sr = record_audio(8, prompt=f'read: "{text}"')
save_recording(sample_name, waveform, sr, text=text)
display(Audio(waveform, rate=sr))

In [ ]:
sample_name = "slovak_example"
text = "Ahoj, letná škola NLP 2026. Vitajte v Kinite."

waveform, sr = record_audio(8, prompt=f'read: "{text}"')
save_recording(sample_name, waveform, sr, text=text)
display(Audio(waveform, rate=sr))

Re-run the cell above (or copy it) with a new `sample_name`/`text` to record more — for example:

```python
sample_name = "slovak_numbers"
text = "Dvadsiateho augusta dvetisícdvadsaťšesť, tridsať eur."
```

Preview everything you've recorded so far (same idea as Section 2's preview):

In [ ]:
for name, (waveform, sr) in my_recordings.items():
    print(f"{name}: {my_transcripts.get(name, '(no reference text saved)')}")
    display(Audio(waveform, rate=sr))

Want to keep your recordings after the runtime resets (e.g. good takes worth adding to the shared bank later)? This zips every recording plus a `transcripts.json` (same format the common bank uses) and triggers one browser download to your PC — nothing touches Drive:

In [ ]:
download_recordings()

Want to start over (e.g. between takes, or to free up memory)? This deletes every in-session recording — there's no undo:

In [ ]:
clear_recordings()

## 4. Load models

Two ASR models to compare:

- **`openai/whisper-large-v3`** — the original, multilingual Whisper checkpoint (handles Slovak, English, and 90+ other languages, but Slovak is a small slice of its training data).
- **[`kinit/whisper-large-v3-turbo-sk`](https://huggingface.co/kinit/whisper-large-v3-turbo-sk)** — Kinit's Whisper-large-v3-**turbo** fine-tuned specifically on Slovak (Common Voice), reporting 9.29% WER vs. 29.23% for the un-tuned turbo model on the same Slovak test set. It's Slovak-only, though — fine-tuning on one language measurably degrades performance on everything else, so don't point it at English or other-language audio.

Both load into an `asr_pipelines` dict below, keyed by the names you'll pick from in Section 5.

In [ ]:
import torch
from transformers import pipeline

MODELS = {
    "whisper-large-v3": "openai/whisper-large-v3",
    "whisper-large-v3-turbo-sk": "kinit/whisper-large-v3-turbo-sk",
}

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
pipeline_device = 0 if device.startswith("cuda") else -1
print(device)

asr_pipelines = {
    name: pipeline(
        "automatic-speech-recognition",
        model=model_id,
        torch_dtype=dtype,
        device=pipeline_device,
    )
    for name, model_id in MODELS.items()
}

## 5. Transcribe: pick a sample + model

`transcribe(sample_name, model_name, language=None, task="transcribe", reference_text=None)` ties everything together:

- `sample_name` — a filename from the common bank (Section 2's printed list, e.g. `"some_clip.wav"`) or a name you gave `save_recording()` in Section 3 (e.g. `"slovak_example"`, no extension — those live in memory, not as files).
- `model_name` — a key from `MODELS` (Section 4): `"whisper-large-v3"` or `"whisper-large-v3-turbo-sk"`.
- `language` — the spoken language of the sample (e.g. `"slovak"`, `"english"`). Required for the Slovak fine-tune; leave as `None` to let the original model auto-detect.
- `task` — `"transcribe"` (stays in the spoken language) or `"translate"` (always converts to English — only the original multilingual model supports this).
- `reference_text` — the known-correct transcript, to print WER/CER alongside the result. Auto-filled by filename/name from `transcripts.json` (Section 2, common bank) or `my_transcripts` (Section 3, your own recordings — populated by `save_recording(..., text=...)`); pass it explicitly only to override.

`compare(sample_name, ...)` takes the same arguments and runs every model in `MODELS` on the same sample back to back, so the WER/CER contrast is visible directly instead of split across separate cells.

In [ ]:
def transcribe(sample_name, model_name, language=None, task="transcribe", reference_text=None):
    waveform, sr = load_sample(sample_name)
    display(Audio(waveform, rate=sr))

    generate_kwargs = {"task": task}
    if language:
        generate_kwargs["language"] = language

    result = asr_pipelines[model_name](
        {"array": to_16k(waveform, sr), "sampling_rate": 16000},
        generate_kwargs=generate_kwargs,
    )
    hypothesis = result["text"]
    label = "Transcribed" if task == "transcribe" else "Translated to English"
    print(f"{label}: {hypothesis}")

    # WER/CER only makes sense when comparing against a reference in the same
    # language as the output - i.e. transcription, not translation.
    reference = reference_text if reference_text is not None else (my_transcripts.get(sample_name) or transcripts.get(sample_name))
    if reference and task == "transcribe":
        wer, cer = word_error_rates(reference, hypothesis)
        print(f"Reference:  {reference}")
        print(f"WER: {wer:.1%}   CER: {cer:.1%}")

    return hypothesis


def compare(sample_name, language=None, task="transcribe", reference_text=None):
    """Run every model in MODELS on the same sample, back to back."""
    for model_name in MODELS:
        print(f"--- {model_name} ---")
        transcribe(sample_name, model_name, language=language, task=task, reference_text=reference_text)
        print()

In [ ]:
# Same Slovak clip through both models - the fine-tune should come out ahead on WER/CER.
compare("slovak_example", language="slovak")

In [ ]:
# Same English clip through both models - the Slovak-only fine-tune should visibly struggle.
compare("english_example", language="english")

### Multilingual clean baselines (common bank)

One clean, single-speaker clip per language from the common bank (Section 2) — no noise or edge cases, just "how well does Whisper handle this language at all." Since `whisper-large-v3-turbo-sk` is Slovak-only, these all run through the original multilingual model.

Each language below is two cells: **load + play**, then **transcribe** (prints the transcript, and WER/CER once `transcripts.json` is in the Drive folder — see the prep notes at the end).

#### English

LibriSpeech `test-clean` — the same audiobook clip already used as a sanity check elsewhere in this repo.

In [ ]:
waveform, sr = load_sample("en_librispeech.wav")
display(Audio(waveform, rate=sr))

In [ ]:
transcribe("en_librispeech.wav", "whisper-large-v3", language="english")

#### Spanish

FLEURS (`es_419`) — a native speaker reading a single news-style sentence.

In [ ]:
waveform, sr = load_sample("es_fleurs.wav")
display(Audio(waveform, rate=sr))

In [ ]:
transcribe("es_fleurs.wav", "whisper-large-v3", language="spanish")

#### French

FLEURS (`fr_fr`). The original upload was very quiet and has been peak-normalized here — if it still sounds noisy when boosted, swap in `fr_fleurs_alt1.wav` or `fr_fleurs_alt2.wav` from the same Drive folder instead (same two-cell pattern, just change the filename).

In [ ]:
waveform, sr = load_sample("fr_fleurs.wav")
display(Audio(waveform, rate=sr))

In [ ]:
transcribe("fr_fleurs.wav", "whisper-large-v3", language="french")

#### German

FLEURS (`de_de`).

In [ ]:
waveform, sr = load_sample("de_fleurs.wav")
display(Audio(waveform, rate=sr))

In [ ]:
transcribe("de_fleurs.wav", "whisper-large-v3", language="german")

### Translate too

Same mechanism, just flip `task` to `"translate"` — only the original multilingual model supports it (the Slovak fine-tune was trained purely for transcription):

In [ ]:
transcribe("slovak_example", "whisper-large-v3", language="slovak", task="translate")

## Common ASR failure modes

Even strong models struggle with — some of the common sample bank in Section 2 is chosen to showcase these live:

- **Named entities** — personal and place names, especially ones from a different language than the speech (e.g. Slovak surnames spoken in an English sentence, or vice versa).
- **Company/brand/product names** — anything newer, niche, or that sounds like an ordinary word, since the model has no domain knowledge to disambiguate from acoustics alone.
- **Numbers, dates, currency, units** — spoken-to-written conversion is inherently ambiguous ("twenty twenty-six" vs "2026"). This is exactly what the DER/DSER metrics in the `slovak-llm-audio` benchmark measure separately from word-level CER/WER.
- **Code-switching** — switching languages mid-sentence (e.g. a Slovak speaker dropping in English technical terms), since most models assume one language per utterance.
- **Low-resource languages** — bigger accuracy gaps for languages with less training data, like Slovak vs English — the reason a dedicated Slovak benchmark exists at all.
- **Homophones & rare words** — usually disambiguated by context, but that safety net is weaker for uncommon vocabulary.
- **Background noise, overlapping speech, multiple speakers** — crosstalk and noisy environments degrade accuracy sharply, and go beyond ASR into diarization.
- **Accents and dialects** — non-native or regional accents underrepresented in training data.
- **Hallucination on silence** — a well-documented Whisper-specific failure: near-silent audio can produce fabricated text instead of an empty transcript.
- **Disfluencies** — filler words, stutters, false starts; models may transcribe them literally, drop them inconsistently, or get confused by them.
- **Punctuation & capitalization** — there's no single "correct" punctuation for spoken language, so this is a common source of apparent errors that aren't really wrong words.

## Presenter prep notes (not part of the live demo)

### Immediate TODOs in the shared Drive folder

- [ ] **Rename `transcripts_candidates.json` → `transcripts.json`.** Section 2 only looks for the latter, so right now WER/CER isn't showing for any common-bank clip even though the reference text is already uploaded.
- [ ] **Prune duplicates/unused candidates.** The folder currently has all 12 WIP files: `en_librispeech_1.wav`–`en_librispeech_6.wav` (extra English book samples — `en_librispeech.wav` is the one actually used in Section 5) and `fr_fleurs_alt1.wav`/`fr_fleurs_alt2.wav` (French backups, in case `fr_fleurs.wav` still sounds off after normalization). Keep whichever you like for variety, delete the rest — and update Section 5's cells / `transcripts.json` if you swap the "official" French clip.

### Samples to record/curate for the common bank

Add each clip's audio file plus its exact reference text to `transcripts.json` in the shared Drive folder (`{"filename.wav": "exact reference text", ...}`) so Section 5 prints WER/CER for it automatically.

- [x] Clean baselines for English, Spanish, French, German — done, see the "Multilingual clean baselines" cells in Section 5.
- [ ] **Clean Slovak baseline** — clear, well-articulated Slovak sentence. Expect low WER on `whisper-large-v3-turbo-sk`, higher on `whisper-large-v3` — the headline comparison for the workshop. (Your own recording in Section 3 already covers this live, but a common-bank version means it's not dependent on a mic working on the day.)
- [ ] **Code-switching** — a Slovak sentence with English technical terms mixed in (e.g. "budeme robiť fine-tuning na GPU cez PyTorch").
- [ ] **Numbers / dates / units** — spoken numbers, currency, dates ("dvadsiateho augusta dvetisícdvadsaťšesť", "tridsať eur") — ties to the DER/DSER note below.
- [ ] **Named entities** — Slovak surnames, foreign company/brand names spoken in a Slovak sentence.
- [ ] **Background noise / multiple speakers** — a clip with noticeable background noise or crosstalk.
- [ ] **Near-silence** — a few seconds of near-silent audio, to try to trigger Whisper's silence-hallucination bug.
- [ ] **Accented speech** — a non-native Slovak or English accent, if you can source one.
- [ ] **Disfluencies** — filler words, false starts, self-corrections, left in rather than edited out.

### Comparisons to run live

- `compare("slovak_example", ...)` and `compare("english_example", ...)` in Section 5 already show the headline fine-tune-vs-original contrast on your own voice, with WER/CER numbers.
- The "Multilingual clean baselines" cells show breadth (Whisper handles 4 languages reasonably), which sets up the failure-mode clips as the contrast: "clean audio is easy, here's what actually breaks it."
- Run `compare(...)` (or just `transcribe(...)` with the multilingual model) on each failure-mode clip above once it's in the bank, and use the printed WER/CER plus the failure-modes discussion to talk through *why* each one is hard.
- The translate demo (Section 5) is currently one-directional (Slovak → English); consider also translating the code-switching clip, to show how translation handles (or mangles) the embedded English.
- Stretch idea, not implemented: time each `asr_pipelines[...]` call to compare inference speed between the two models — ties into the `slovak-llm-audio` benchmark repo's RTFX metric, if you want to make a speed-vs-accuracy point too.